<a href="https://colab.research.google.com/github/mdruwaid/college-labs/blob/main/wrod_embeding_using_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:


!pip install -U spacy
!python -m spacy download en_core_web_sm


import spacy
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim

nlp = spacy.load("en_core_web_sm")

corpus = """
Neural networks are very useful for natural language processing.
They can learn word embeddings from raw text by training on large corpora.
"""

doc = nlp(corpus.lower())
tokens = [token.text for token in doc if token.is_alpha]
print (tokens)

vocab = list(set(tokens))
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for word, i in word_to_ix.items()}
vocab_size = len(vocab)

def generate_skipgram_pairs(tokens, window_size=2):
    pairs = []
    for idx, word in enumerate(tokens):
        for neighbor in range(max(idx - window_size, 0),
                              min(idx + window_size + 1, len(tokens))):
            if neighbor != idx:
                pairs.append((word, tokens[neighbor]))
    return pairs

training_data = generate_skipgram_pairs(tokens)
print("Sample Skip Gram Pairs:",training_data[:5])

class Word2Vec(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(Word2Vec, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.output_layer = nn.Linear(embedding_dim, vocab_size)

    def forward(self, center_word_idx):
        embeds = self.embeddings(center_word_idx)
        out = self.output_layer(embeds)
        return out

embedding_dim = 50
model = Word2Vec(vocab_size, embedding_dim)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

def word_to_tensor(word):
    return torch.tensor([word_to_ix[word]], dtype=torch.long)

for epoch in range(100):
    total_loss = 0

    for center, context in training_data:
        center_tensor = word_to_tensor(center)
        context_tensor = word_to_tensor(context)

        optimizer.zero_grad()

        output = model(center_tensor)
        loss = loss_fn(output, context_tensor)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

def get_embedding(word):
    word_index = word_to_ix[word]
    vector = model.embeddings(torch.tensor([word_index])).detach().numpy()
    return vector

embedding = get_embedding("neural")
print("Embedding for 'neural':\n", embedding)



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
['neural', 'networks', 'are', 'very', 'useful', 'for', 'natural', 'language', 'processing', 'they', 'can', 'learn', 'word', 'embeddings', 'from', 'raw', 'text', 'by', 'training', 'on', 'large', 'corpora']
Sample Skip Gram Pairs: [('neural', 'networks'), ('neural', 'are'), ('networks', 'neural'), ('networks', 'are'), ('networks', 'very')]
Epoch 10, Loss: 154.5055
Epoch 20, Loss: 138.2696
Epoch 30, Loss: 133.6279
Epoch 40, Loss: 131.5514
Epoch 50, Loss: 130.3961
Epoch 60, Loss: 129.6645
Epoch 70, Loss: 129.1595
Epoch 80, Loss: 128.7885
Epoch 90, Loss: 128.5025
Epoch 100, Loss: 1